# Day 018 Project Solution — Token Dashboard Chatbot

In [ ]:
import ollama

def extract_usage(response: dict) -> dict:
    return {
        'input_tokens':  response.get('prompt_eval_count', 0),
        'output_tokens': response.get('eval_count', 0),
        'duration_ms':   response.get('eval_duration', 0) // 1_000_000,
    }

def tokens_per_second(response: dict) -> float:
    duration_s = response.get('eval_duration', 0) / 1e9
    if duration_s == 0:
        return 0.0
    return response.get('eval_count', 0) / duration_s

class UsageTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.call_count = 0

    @property
    def total_tokens(self) -> int:
        return self.total_input + self.total_output

    def record(self, response: dict) -> None:
        usage = extract_usage(response)
        self.total_input  += usage['input_tokens']
        self.total_output += usage['output_tokens']
        self.call_count   += 1

PRICES = {
    'gpt-4o':            {'input': 2.50,  'output': 10.00},
    'claude-3-5-sonnet': {'input': 3.00,  'output': 15.00},
    'gemini-1.5-pro':    {'input': 1.25,  'output': 5.00},
}

def cost_estimate(input_tokens, output_tokens, model='gpt-4o'):
    if model not in PRICES:
        raise ValueError(f'Unknown model {model!r}. Choose from: {list(PRICES)}')
    p = PRICES[model]
    input_cost  = input_tokens  * p['input']  / 1_000_000
    output_cost = output_tokens * p['output'] / 1_000_000
    return {
        'model':       model,
        'input_cost':  round(input_cost,  6),
        'output_cost': round(output_cost, 6),
        'total_cost':  round(input_cost + output_cost, 6),
    }

class TokenAwareChatbot:
    def __init__(self, model='llama3.2',
                 system_prompt='You are a helpful assistant.'):
        self.model = model
        self.tracker = UsageTracker()
        self._history = [{'role': 'system', 'content': system_prompt}]
        self._last_response = None

    def chat(self, user_input: str) -> str:
        self._history.append({'role': 'user', 'content': user_input})
        response = ollama.chat(model=self.model, messages=self._history)
        self._last_response = response
        self.tracker.record(response)
        reply = response['message']['content']
        self._history.append({'role': 'assistant', 'content': reply})
        return reply

    def summary(self) -> str:
        return (
            f'Calls: {self.tracker.call_count} | '
            f'Input: {self.tracker.total_input} tok | '
            f'Output: {self.tracker.total_output} tok | '
            f'Total: {self.tracker.total_tokens} tok'
        )

SYSTEM_PROMPT = 'You are a helpful assistant. Be concise.'

# Scripted demo — three turns, no input()
chatbot = TokenAwareChatbot(model='llama3.2', system_prompt=SYSTEM_PROMPT)

turns = [
    'What is 15 * 17?',
    'What is a token in the context of language models, in one sentence?',
    'Give me a one-sentence tip for keeping prompts short.',
]

for user_input in turns:
    print(f'You: {user_input}')
    reply = chatbot.chat(user_input)
    usage = extract_usage(chatbot._last_response)
    tps = tokens_per_second(chatbot._last_response)
    print(f'Bot: {reply}')
    print(f'[Tokens: {usage["input_tokens"]} in / {usage["output_tokens"]} out | {tps:.1f} tok/s]')
    print()

print('=' * 60)
print('SESSION SUMMARY')
print(chatbot.summary())
est = cost_estimate(chatbot.tracker.total_input, chatbot.tracker.total_output, 'gpt-4o')
print(f'If using gpt-4o: ${est["total_cost"]:.6f}')
est2 = cost_estimate(chatbot.tracker.total_input, chatbot.tracker.total_output, 'claude-3-5-sonnet')
print(f'If using claude-3-5-sonnet: ${est2["total_cost"]:.6f}')
